# Text Splitting for RAG - My Experiment

This notebook demonstrates the **data transformation** step of a Retrieval-Augmented Generation (RAG) pipeline:

1. **Install dependencies** - get the PDF loader and text splitter libraries.
2. **Load a PDF** - read pages and extract raw text.
3. **Split text into chunks** - convert long pages into smaller, overlapping pieces that an LLM can process.
4. **Inspect the result** - verify how many chunks were produced and what each chunk contains.

Run the cells top to bottom. Markdown cells explain what each code block does and why it matters.


## Section 1: Environment Setup

Before loading and splitting documents we need the right Python packages.

- `pypdf` (or `PyPDF2`) reads text from PDF files.
- `langchain-pypdf` was an experimental LangChain loader at the time this notebook was written.

> **Note:** The cell below tries to install from a Git URL. The captured output shows a repository-not-found error, which is preserved exactly as it happened. In a fresh environment you can use `pip install pypdf langchain-text-splitters` instead.


In [3]:
# !pip install pypdf2
# !pip install pypdf
!pip install git+https://github.com/langchain-ai/langchain-pypdf.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/langchain-ai/langchain-pypdf.git to C:\Users\BhargavDharan\AppData\Local\Temp\pip-req-build-dctbzn7n


  Running command git clone --filter=blob:none --quiet https://github.com/langchain-ai/langchain-pypdf.git 'C:\Users\BhargavDharan\AppData\Local\Temp\pip-req-build-dctbzn7n'
  remote: Repository not found.
  fatal: repository 'https://github.com/langchain-ai/langchain-pypdf.git/' not found
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/langchain-ai/langchain-pypdf.git 'C:\Users\BhargavDharan\AppData\Local\Temp\pip-req-build-dctbzn7n' did not run successfully.
  │ exit code: 128
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Failed to build 'git+https://github.com/langchain-ai/langchain-pypdf.git' when git clone --filter=blob

## Section 2: Loading the PDF Document

RAG systems need source documents. Here we use `pypdf.PdfReader` to extract text from a local PDF named `rag_sample_document.pdf`.

**Why this step matters:** LLMs cannot read PDFs directly. We must pull the text out first and store it in a structure the rest of the pipeline can use.

**What the code does, line by line:**
- `PdfReader("rag_sample_document.pdf")` opens the PDF.
- `reader.pages` gives us every page object.
- The `for` loop iterates over pages, extracts text, and skips empty pages.
- Each page is stored as a dictionary with `page_content` (the text) and `metadata` (the page number).
- Finally we print how many pages were loaded and preview the first 500 characters.

**Expected output:**
- `Loaded 1 pages` tells us the PDF has one extractable page.
- The preview shows the title and first few sentences of the document.


In [5]:
from pypdf import PdfReader

reader = PdfReader("rag_sample_document.pdf")
docs = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        docs.append({
            "page_content": text,
            "metadata": {"page": i + 1}
        })

print(f"Loaded {len(docs)} pages")
print(docs[0]["page_content"][:500])

Loaded 1 pages
An Introduction to Retrieval-Augmented Generation
(RAG)
Document ID: TECH-RAG-2026-001 | Classification: Public
Retrieval-Augmented Generation (RAG) is an advanced architectural framework designed to optimize the
output of Large Language Models (LLMs) by sourcing authoritative data from external knowledge bases
before generating a response. Traditional language models rely solely on internal parametric knowledge
acquired  during  their  initial  training  phases,  which  often  renders  them  su


## Section 3: Splitting Text into Chunks

A single PDF page can be thousands of characters long. Most embedding models and LLMs work best with smaller passages, so we split the text into **chunks**.

**Why chunking is needed:**
- Embedding models have token limits.
- Smaller chunks retrieve more precise results during similarity search.
- Overlapping chunks keep context across boundaries so sentences are not cut off mid-thought.

**What the code does:**
- `RecursiveCharacterTextSplitter` is a LangChain splitter that breaks text at natural boundaries (paragraphs, sentences, words) until each chunk is below `chunk_size`.
- `chunk_size=1000` means each chunk aims for ~1000 characters.
- `chunk_overlap=200` means 200 characters are shared between consecutive chunks to preserve context.
- We wrap our dictionaries in LangChain `Document` objects because `split_documents()` expects that type.
- `final_documents` is the list of chunked `Document` objects we can later embed and store in a vector database.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# If using direct pypdf approach, wrap in Document objects:
from langchain_core.documents import Document

langchain_docs = [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in docs]
final_documents = text_splitter.split_documents(langchain_docs)

## Section 4: Inspecting the PDF and the Split Results

It is good practice to verify what the loader produced before moving on to embeddings. The next few cells print diagnostics:
- total number of pages in the PDF
- character count per page
- whether any page has no extractable text
- how many chunks the splitter created
- the content of the first chunk


### 4.1 Verify page-level extraction

This cell is a diagnostic check. It re-opens the PDF and prints:
- `Total pages in PDF`
- For each page, the number of characters extracted
- A warning if a page is blank
- A short preview of the first 100 characters

Use this when a PDF seems to produce unexpected chunks or empty results.


In [ ]:
from pypdf import PdfReader

reader = PdfReader("rag_sample_document.pdf")

print(f"Total pages in PDF: {len(reader.pages)}")
print("-" * 50)

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    print(f"Page {i+1}: {len(text) if text else 0} characters")
    if not text or len(text.strip()) == 0:
        print(f"  ⚠️  Page {i+1} has NO extractable text")
    else:
        print(f"  ✓  First 100 chars: {text[:100]}")

### 4.2 Count the chunks

`len(final_documents)` tells us how many chunks the splitter produced from the loaded pages.


In [ ]:
len(final_documents)

### 4.3 Inspect the first chunk

`final_documents[0]` displays the first chunk. Each chunk is a LangChain `Document` containing:
- `page_content`: the actual text of the chunk
- `metadata`: extra information carried forward, such as the original page number

This confirms that splitting preserved the page metadata we attached during loading.


In [ ]:
final_documents[0]